[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UoA-eResearch/embedding-llms-qualitative-data-workshop/blob/main/notebooks/01-environment-setup.ipynb)

# Notebook 01 — Environment Setup

**Duration:** ~20 minutes

Welcome. In this notebook you will set up the tools we use throughout the workshop.

By the end you will have:
- Python running in your browser
- A connection to a large language model (LLM) — an AI system that can read and generate text
- Enough confidence to keep going

You do not need to install anything on your computer. Everything runs here in **Google Colab** — a free coding environment that works entirely in your web browser. 

## What is a cell?

This notebook is made of **cells**. Each cell is a block that contains either text (like this one) or code (like the one below).

To run a code cell:
1. Click on it so it is highlighted
2. Press **Shift + Enter** on your keyboard (or click the ▶ play button on the left)

The result appears directly below the cell. If something goes wrong, you will see an error message — that is normal. Read the last line of the error. It usually tells you what happened.

Let's try it.

## Exercise 1 — Python as a calculator

Python can do arithmetic. Run the cell below to see it work.

In [2]:
2 + 3

5

You should see the number `5` below the cell.

Now try some of these. Change the numbers, or write your own. Run the cell each time to see the result.

In [3]:
# Try any of these — change the numbers, or write your own.
# Delete the # at the start of a line to "uncomment" it and make it run.

# 10 * 5
# 100 / 3
# 2 ** 10        # ** means "to the power of"
# (4 + 6) * 2

Lines that start with `#` are **comments**. They are notes for us to understand what the code is doing, Python ignores them.

## Exercise 2 — Variables and print()

A **variable** stores a value so you can use it later. Think of it as a labelled box: you put something in, and refer to it by name.

In [ ]:
name = "Kyle"

In [ ]:
name

In [ ]:
# The `print()` function displays a value on screen. Run this cell to see both in action.
print("Hello, my name is", name)
print("Welcome to the workshop.")

Your turn. Change `"____"` to your own name and run the cell.

In [ ]:
name = "____"    # <-- replace ____ with your name (keep the quotes)
print("Hello my name is", name)

# You can store numbers too:
sections = 200
print("The Privacy Act 2020 has roughly", sections, "sections.")
print("Reading them all would take a while. That is why we have an LLM.")

Hello, ____
The Privacy Act 2020 has roughly 200 sections.
Reading them all would take a while. That is why we have an LLM.


Notice: text goes in quotes (`"like this"`), numbers do not. If you see an error about quotes, check that both the opening and closing quotes are there.

## Where this fits in a real research workflow

> **Research context:** Everything we do today follows a workflow used in a published study that computationally analysed New Zealand's Mental Health Act (Ardekani et al., 2026). Step 1 of that workflow is **ingestion** — grabbing the online stored files the NZ government publishes for every Act of Parliament. By the end of this notebook, you will have exactly the infrastructure needed for that step.

The six-step workflow from the paper is:

| Step | What it does | When we do it |
|------|-------------|---------------|
| 1. Ingestion | Parse raw data into a usable format | **This notebook** |
| 2. Extraction | Pull out structured information | Notebook 03 |
| 3. Expansion | Build outward from a starting point | Discussed in 03 |
| 4. Semantic enrichment | Use an LLM to summarise and clean | Notebook 04 |
| 5. Analysis | Find patterns and themes | Notebook 04 |
| 6. Interpretation | Check findings against reality | Notebooks 04 and 05 |

Keep this table in mind. We will come back to it at the end of the day.

## Setting up our tools

For the rest of the workshop we need a few tools:

| Tool | What it does |
|------|--------------|
| `groq` | Connects Python to an LLM (we use the free Groq API) |
| `requests` | Fetches data from the internet (we will download legislation) |
| `lxml` | Reads XML files — the format NZ legislation is published in |
| `Pillow` | Works with images (used in notebook 05) |

An **API key** is a password that identifies you to the service. You will create your own in the next step.

<details>
<summary><strong>What is Groq? What is an API? How do the pieces fit together?</strong></summary>

**Groq** is a company that hosts large language models and lets you use them for free through their API. The models themselves (like Llama 3.3) are built by Meta — Groq just runs them on fast hardware so we can send questions and get answers back in seconds.

An **API** (Application Programming Interface) is a way for one program to talk to another over the internet. Instead of opening a chatbot in your browser, our Python code sends a question directly to the model and receives a structured answer back.

Here is how the pieces fit together:

```
┌─────────────┐       ┌──────────────┐       ┌──────────────────┐
│  Your       │       │  Google      │       │  Groq servers    │
│  browser    │ ───── │  Colab       │ ───── │  (runs the LLM)  │
│             │       │  (runs your  │  API  │                  │
│             │       │   Python)    │ call  │  Llama 3.3 model │
└─────────────┘       └──────────────┘       └──────────────────┘
                            │                        │
                       You write code           Model generates
                       and press Run            a response
                            │                        │
                            └───── answer arrives ───┘
```

Everything in this workshop happens in the middle box (Colab). Your browser is just the window. The LLM lives on Groq's servers — your code sends it a question over the internet, and the answer comes back.

</details>

## Get your Groq API key

Follow these steps. It takes about two minutes.

1. Go to [console.groq.com/keys](https://console.groq.com/keys)
2. Click **Sign in with Google** and use your university Google account
3. Once signed in, click **Create API Key**
4. Give it any name (e.g. "workshop")
5. **Copy the key** — it starts with `gsk_` and is a long string of letters and numbers

**Important:** 
- The key only appears once. If you lose it, delete it and create a new one.
- Your API key is a password. Do not share it with anyone or post it online. 

## Store your key in Colab Secrets

Instead of pasting your key into every notebook, we store it once using Colab's built-in **Secrets** feature. This keeps it safe and makes it available across all notebooks automatically.

1. In the left sidebar, click the **key icon** (🔑 Secrets)
2. Click **Add new secret**
3. Set the **Name** to exactly: `GROQ_API_KEY`
4. Paste your key into the **Value** field
5. Toggle **Notebook access** to **On**

Once saved, every notebook in this workshop can read the key automatically — you will not need to paste it again.

In [ ]:
# ============================================================
# SETUP CELL — Run this once at the start of every notebook
# ============================================================

!pip install groq requests lxml Pillow
import os, json, base64, requests, io
from groq import Groq
from lxml import etree
from PIL import Image
from IPython.display import Image as IPImage, display

# Load API key from Colab Secrets (set up above)
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    print("API key loaded from Colab Secrets.")
except Exception:
    os.environ["GROQ_API_KEY"] = "paste_your_key_here"   # <-- fallback: paste key here
    print("Could not load from Secrets. Paste your key in the line above.")

client = Groq(api_key=os.environ["GROQ_API_KEY"])
TEXT_MODEL = "llama-3.3-70b-versatile"
VISION_MODEL = "meta-llama/llama-4-maverick-17b-128e-instruct"

print("Setup complete.")


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Could not load from Secrets. Paste your key in the line above.
Setup complete.


You should see `API key loaded from Colab Secrets.` followed by `Setup complete.`

<details>
<summary><strong>Got an error? Click here for troubleshooting.</strong></summary>

If you see `Could not load from Secrets`, check:
- Did you add the secret with the exact name `GROQ_API_KEY`?
- Did you toggle **Notebook access** to On?
- Is the key complete? (It should start with `gsk_`)

If Secrets is not working, you can paste the key directly into the fallback line in the setup cell as a workaround.

</details>

## Exercise 3 — Test your LLM connection

Let's confirm everything works by sending a short message to the AI model and reading its reply. 

We will then go through the message syntax to understand how it works.

Run the cell below. It asks the model a simple question.

In [14]:
response = client.chat.completions.create(
    model=TEXT_MODEL,
    messages=[
        {"role": "user", "content": "In one sentence, what is the Privacy Act 2020?"}
    ]
)

print(response.choices[0].message.content)

The Privacy Act 2020 is a New Zealand law that governs the collection, use, and disclosure of personal information, providing individuals with greater control over their personal data and imposing obligations on agencies to protect and handle it in a responsible manner.


You should see a one-sentence answer about New Zealand's Privacy Act. The exact wording may differ from your neighbour's — that is expected. We will explore why in the next notebook.

Let's break down what just happened:

| Line | What it does |
|------|--------------|
| `response` | Stores the entire response from the API |
| `client.chat.completions.create(...)` | Sends a message to the LLM via the Groq API |
| `model=TEXT_MODEL` | Tells it which model to use (we set this in the setup cell) |
| `messages=[{...}]` | The conversation — a list of messages, each with a `role` and `content` |
| `"role": "user"` | Labels who is speaking — `"user"` means you, the human asking the question |
| `"content": "..."` | The actual text you are sending |
| `response.choices[0].message.content` | Extracts the model's reply from the response |

You will see this pattern in every notebook. The only thing that changes is the message you send.

### Your turn

Change the question below to anything you like and run the cell. Try asking it something related to your own research area.

In [12]:
# Change the question to anything you like.
# Some ideas:
#   "What is thematic analysis in qualitative research?"
#   "Summarise the purpose of the NZ Human Rights Act 1993 in two sentences."
#   "What are the ethical risks of using AI in social science research?"

my_question = "What is thematic analysis in qualitative research?"

response = client.chat.completions.create(
    model=TEXT_MODEL,
    messages=[
        {"role": "user", "content": my_question}
    ]
)

print(response.choices[0].message.content)

Thematic analysis is a widely used qualitative research method that involves identifying, analyzing, and interpreting patterns and themes within a dataset. The goal of thematic analysis is to uncover underlying meanings, concepts, and ideas that emerge from the data, and to organize them into a coherent and meaningful framework.

The process of thematic analysis typically involves the following steps:

1. **Data preparation**: The researcher prepares the data by transcribing interviews, observations, or other forms of qualitative data.
2. **Familiarization**: The researcher reads and becomes familiar with the data to get a sense of the overall content and tone.
3. **Coding**: The researcher codes the data by assigning labels or tags to specific segments of the data. This helps to identify initial patterns and themes.
4. **Theme identification**: The researcher identifies themes by looking for patterns, relationships, and meanings within the coded data.
5. **Theme refinement**: The rese

### Now add context with a persona

The API supports a second role called `"system"`. This message is not a question — it tells the model *who it is* before it reads your question. Think of it as giving the model a job title.

Run the cell below. It sends the **same question**, but now the model has been told it is a New Zealand legal assistant. Compare the answer to the one above — is it different?

In [ ]:
# Same question, but now with a system message that sets a persona
response = client.chat.completions.create(
    model=TEXT_MODEL,
    messages=[
        {"role": "system", "content": "You are a New Zealand legal assistant."},
        {"role": "user", "content": my_question}
    ]
)

print(response.choices[0].message.content)

The `"system"` message shaped the answer without changing the question. This is how we add context and control to our prompts. We will use this extensively from notebook 02 onward.

## What you accomplished

In this notebook you:

- Ran Python code in your browser using Google Colab
- Learned what cells, variables, and `print()` do
- Installed the libraries we use throughout the workshop
- Created a Groq API key and connected to an LLM
- Sent your first message to the model and read its reply

This is the ingestion infrastructure — Step 1 of the research workflow we introduced earlier. Everything else builds on what you just set up.

**Next up:** In notebook 02, we look more closely at how the model behaves — why it sometimes gives different answers to the same question, and how the way you phrase a question changes what you get back.